# Receipt Schema Prototype

Exploratory notebook that reverse-engineers a grammar for CORD receipts — which tags occur, which are pure `str` vs mixed `str`/`list`, and how to normalize raw labels into one consistent shape. The decisions made here were later hand-coded into `utils/_receipt_schema.py` (the pydantic `Receipt` grammar) and `utils/_cord_preprocess.py` (`preprocess_cord`).

# 0. Setup: Install Requirements

Install pydantic, used to express and validate the receipt schema.

In [1]:
!pip install pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pydantic]3/4 [pydantic]


# 1. Load CORD-v2

Load all three splits of the CORD-v2 receipt dataset.

In [4]:
import json
from collections import Counter, defaultdict
import pandas as pd
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2")
print({split: len(ds[split]) for split in ds})

{'train': 800, 'validation': 100, 'test': 100}


# 2. Tag Frequency & Type Breakdown

Walk every label across all splits, counting how often each dotted tag appears and which value types it takes (`str` / `list` / `dict`). The type columns reveal which fields are pure-`str` and which genuinely flip between `str` and `list` (the "mixed" fields).

In [5]:
tag_counts = Counter()
type_obs = defaultdict(Counter)          # tag -> Counter({"str": n, "list": n, "dict": n})

def walk(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            path = f"{prefix}.{k}" if prefix else k
            tag_counts[path] += 1
            type_obs[path][type(v).__name__] += 1
            walk(v, path)
    elif isinstance(obj, list):
        for item in obj:                 # list items share the parent tag namespace
            walk(item, prefix)

for split in ds:
    for row in ds[split]:
        gt = json.loads(row["ground_truth"])["gt_parse"]
        walk(gt, "")

rows = []
for tag, n in tag_counts.items():
    t = type_obs[tag]
    rows.append({
        "tag": tag, "count": n,
        "n_str": t.get("str", 0),
        "n_list": t.get("list", 0),
        "n_dict": t.get("dict", 0),
    })

df = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
df["is_mixed"] = (df["n_str"] > 0) & (df["n_list"] > 0)   # fields that flip str<->list
pd.set_option("display.max_rows", None)
display(df)

print("\nMixed str/list fields (these justify list-of-str normalization):")
display(df[df["is_mixed"]][["tag", "count", "n_str", "n_list"]])

,tag,count,n_str,n_list,n_dict,is_mixed
0,menu.nm,2569,2565,4,0,True
1,menu.price,2559,2559,0,0,False
2,menu.cnt,2331,2330,1,0,True
3,menu,1000,0,569,431,False
4,total,998,0,0,998,False
5,total.total_price,972,971,1,0,True
6,menu.unitprice,737,737,0,0,False
7,sub_total,681,0,1,680,False
8,sub_total.subtotal_price,663,655,8,0,True
9,total.cashprice,648,640,8,0,True



Mixed str/list fields (these justify list-of-str normalization):


,tag,count,n_str,n_list
0,menu.nm,2569,2565,4
2,menu.cnt,2331,2330,1
5,total.total_price,972,971,1
8,sub_total.subtotal_price,663,655,8
9,total.cashprice,648,640,8
10,total.changeprice,620,619,1
11,sub_total.tax_price,445,439,6
17,total.creditcardprice,151,150,1
19,menu.num,108,107,1
20,menu.discountprice,97,93,4


# 3. Derive the Schema

Turn the frequency table into the two decisions the grammar encodes — which tags are allowed (rare ones pruned away) and which allowed fields must be `List[str]` vs `str`. These map directly onto `utils/_receipt_schema.py`.

## 3.1 Prune Rare Tags (union tag tree)

Keep only tags with `count > MIN` and assemble them into a nested union tree. Dropping a tag here means the schema can never emit it, so this fixes the set of keys allowed in each pydantic model.

In [6]:
MIN = 3
kept = df[df["count"] > MIN]["tag"].tolist()
dropped = df[df["count"] <= MIN][["tag", "count"]]

def build_tree(tags):
    tree = {}
    for t in tags:
        node = tree
        for part in t.split("."):
            node = node.setdefault(part, {})
    return tree

print(f"Kept {len(kept)} tags (count > {MIN}); dropped {len(dropped)}:")
display(dropped)
print(json.dumps(build_tree(kept), indent=2, ensure_ascii=False))

Kept 29 tags (count > 3); dropped 5:


,tag,count
29,menu.vatyn,3
30,void_menu.price,1
31,void_menu.nm,1
32,void_menu,1
33,sub_total.othersvc_price,1


{
  "menu": {
    "nm": {},
    "price": {},
    "cnt": {},
    "unitprice": {},
    "sub": {
      "nm": {},
      "cnt": {},
      "price": {},
      "unitprice": {}
    },
    "num": {},
    "discountprice": {},
    "etc": {},
    "itemsubtotal": {}
  },
  "total": {
    "total_price": {},
    "cashprice": {},
    "changeprice": {},
    "menuqty_cnt": {},
    "creditcardprice": {},
    "menutype_cnt": {},
    "emoneyprice": {},
    "total_etc": {}
  },
  "sub_total": {
    "subtotal_price": {},
    "tax_price": {},
    "service_price": {},
    "etc": {},
    "discount_price": {}
  }
}


## 3.2 Identify Mixed str/list Fields

Collect the fields that appear as both `str` and `list` — these become `List[str]` in the schema — and freeze them to `mixed_fields.json` so train-prep and eval load the exact same set. Container fields (`menu`/`void_menu`) flip `dict`↔`list` instead and are handled separately by the normalizer.

In [7]:
import json

MIXED_FIELDS = frozenset(
    df.loc[(df["n_str"] > 0) & (df["n_list"] > 0), "tag"].tolist()
)
# container fields (menu/void_menu) flip dict<->list, NOT str<->list,
# so they correctly fall OUT of MIXED_FIELDS. Handled separately below.
LIST_CONTAINERS = {"menu", "void_menu"}

print(f"{len(MIXED_FIELDS)} mixed fields -> forced to list-of-str:")
for t in sorted(MIXED_FIELDS):
    print("  ", t)

# persist so train-prep and eval both load the SAME set
with open("mixed_fields.json", "w") as f:
    json.dump(sorted(MIXED_FIELDS), f, indent=2)

12 mixed fields -> forced to list-of-str:
   menu.cnt
   menu.discountprice
   menu.nm
   menu.num
   sub_total.discount_price
   sub_total.etc
   sub_total.subtotal_price
   sub_total.tax_price
   total.cashprice
   total.changeprice
   total.creditcardprice
   total.total_price


# 4. Normalize Labels

Coerce every raw label into the single consistent shape the schema expects, so `Receipt.model_validate` accepts it. This logic later became `preprocess_cord` in `utils/_cord_preprocess.py`.

## 4.1 Define the Normalizer

Type-aware normalization: wrap mixed-field leaf strings into `[str]` (without double-wrapping existing lists), coerce single-dict containers (`menu`/`void_menu`) to a one-element list, and leave dicts, arrays-of-dicts, and non-mixed leaves untouched.

In [8]:
def normalize_value(v, path):
    if isinstance(v, str):
        return [v] if path in MIXED_FIELDS else v
    if isinstance(v, dict):
        return normalize_dict(v, path)
    if isinstance(v, list):
        if all(isinstance(x, str) for x in v):
            return v                                  # leaf list: leave untouched
        return [normalize_value(x, path) for x in v]  # array of dicts: recurse, same path
    return v                                          # int/float/None passthrough

def normalize_dict(d, path=""):
    out = {}
    for k, v in d.items():
        child = f"{path}.{k}" if path else k
        if k in LIST_CONTAINERS and isinstance(v, dict):
            v = [v]                                   # single-item receipt -> wrap container
        out[k] = normalize_value(v, child)
    return out

def normalize_receipt(gt_parse: dict) -> dict:
    return normalize_dict(gt_parse, "")

## 4.2 Verify on a Sample Receipt

Run the normalizer on a hand-built sample and assert the key behaviors (str→`[str]` for mixed fields, existing lists preserved, pure-`str` fields kept as `str`) so a future refactor can't silently regress them.

In [9]:
sample = {
  "menu": [
    {"nm": "NS MINI STICK", "unitprice": "1,200", "cnt": "2X",
     "discountprice": "240-", "price": "2,400"},
    {"nm": "FIXALL HK 26521", "unitprice": "19,900", "cnt": "2X",
     "discountprice": ["8,600 -", "3,120-"], "price": "39,800"},
  ],
  "sub_total": {"subtotal_price": "43,110"},
  "total": {"total_price": "43,110", "creditcardprice": "43,110"},
}
print(json.dumps(normalize_receipt(sample), indent=2, ensure_ascii=False))

# explicit assertions so a future "simplification" can't silently regress this
out = normalize_receipt(sample)
if "menu.discountprice" in MIXED_FIELDS:
    assert out["menu"][0]["discountprice"] == ["240-"]            # str -> [str]
    assert out["menu"][1]["discountprice"] == ["8,600 -", "3,120-"]  # list kept
if "total.total_price" not in MIXED_FIELDS:
    assert out["total"]["total_price"] == "43,110"               # str kept as str
print("OK")

{
  "menu": [
    {
      "nm": [
        "NS MINI STICK"
      ],
      "unitprice": "1,200",
      "cnt": [
        "2X"
      ],
      "discountprice": [
        "240-"
      ],
      "price": "2,400"
    },
    {
      "nm": [
        "FIXALL HK 26521"
      ],
      "unitprice": "19,900",
      "cnt": [
        "2X"
      ],
      "discountprice": [
        "8,600 -",
        "3,120-"
      ],
      "price": "39,800"
    }
  ],
  "sub_total": {
    "subtotal_price": [
      "43,110"
    ]
  },
  "total": {
    "total_price": [
      "43,110"
    ],
    "creditcardprice": [
      "43,110"
    ]
  }
}
OK
